# DataFrames (Week 2 tutorial, notebook 3 of 8)

**What this notebook covers:** the pandas `DataFrame`, a table whose columns are Series that share one index. You will select columns and rows, add and remove columns, filter rows with conditions, change the index, and work with a two-level index (a *MultiIndex*).

**How to run it:** this notebook runs in your browser, so there is nothing to install. Click a code cell and press **Shift + Enter** to run it and move to the next one, or use **Run → Run All Cells**. Run the cells in order, because later cells use variables created earlier. Several cells change `df`, and later cells depend on those changes.

**What to expect:** the random numbers are fixed with `np.random.seed(101)`, so you get exactly the tables shown. If you re-run a cell that draws random numbers, the later numbers change; use **Kernel → Restart Kernel and Run All Cells** to get back to the saved results.

**Next:** *04-Missing Data*.

## What is a DataFrame?

DataFrames are the workhorse of pandas and are directly inspired by the R programming language. We can think of a DataFrame as a bunch of Series objects put together to share the same index. Let's use pandas to explore this topic!

The next cells import the libraries, fix the random seed and build a table of random numbers with rows labelled A to E and columns W to Z. `'A B C D E'.split()` is a short way to write the list `['A', 'B', 'C', 'D', 'E']`.

In [183]:
import pandas as pd
import numpy as np

In [184]:
from numpy.random import randn
# fix the random seed so everyone gets the same random numbers
np.random.seed(101)

In [185]:
# 5 rows x 4 columns of random numbers from a standard normal distribution
df = pd.DataFrame(randn(5,4),index='A B C D E'.split(),columns='W X Y Z'.split())

In [186]:
df

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
C,-2.018168,0.740122,0.528813,-0.589001
D,0.188695,-0.758872,-0.933237,0.955057
E,0.190794,1.978757,2.605967,0.683509


## Selection and Indexing

Let's learn the various methods to grab data from a DataFrame. Square brackets with one column name return that column as a Series; with a list of names they return a smaller DataFrame.

In [187]:
df['W']

A    2.706850
B    0.651118
C   -2.018168
D    0.188695
E    0.190794
Name: W, dtype: float64

In [188]:
# Pass a list of column names
df[['W','Z']]

,W,Z
A,2.706850,0.503826
B,0.651118,0.605965
C,-2.018168,-0.589001
D,0.188695,0.955057
E,0.190794,0.683509


In [189]:
# Attribute access: works for simple column names, but not recommended
# (it fails for names with spaces or names that clash with DataFrame methods)
df.W

A    2.706850
B    0.651118
C   -2.018168
D    0.188695
E    0.190794
Name: W, dtype: float64

DataFrame columns are just Series (`type()` shows the class, `pandas.Series`):

In [190]:
type(df['W'])

pandas.Series

**Creating a new column:** assign values to a new column name. pandas adds `W` and `Y` row by row, matching the rows by their index labels.

In [191]:
df['new'] = df['W'] + df['Y']

In [192]:
df

,W,X,Y,Z,new
A,2.706850,0.628133,0.907969,0.503826,3.614819
B,0.651118,-0.319318,-0.848077,0.605965,-0.196959
C,-2.018168,0.740122,0.528813,-0.589001,-1.489355
D,0.188695,-0.758872,-0.933237,0.955057,-0.744542
E,0.190794,1.978757,2.605967,0.683509,2.796762


**Removing columns**

`axis=1` means columns (`axis=0` means rows). `drop` returns a new DataFrame and leaves `df` unchanged unless you pass `inplace=True`, as the next cells show.

In [193]:
df.drop('new',axis=1)

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
C,-2.018168,0.740122,0.528813,-0.589001
D,0.188695,-0.758872,-0.933237,0.955057
E,0.190794,1.978757,2.605967,0.683509


In [194]:
# Not inplace unless specified!
df

,W,X,Y,Z,new
A,2.706850,0.628133,0.907969,0.503826,3.614819
B,0.651118,-0.319318,-0.848077,0.605965,-0.196959
C,-2.018168,0.740122,0.528813,-0.589001,-1.489355
D,0.188695,-0.758872,-0.933237,0.955057,-0.744542
E,0.190794,1.978757,2.605967,0.683509,2.796762


In [195]:
df.drop('new',axis=1,inplace=True)

In [196]:
df

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
C,-2.018168,0.740122,0.528813,-0.589001
D,0.188695,-0.758872,-0.933237,0.955057
E,0.190794,1.978757,2.605967,0.683509


Can also drop rows this way:

In [197]:
df.drop('E',axis=0)

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
C,-2.018168,0.740122,0.528813,-0.589001
D,0.188695,-0.758872,-0.933237,0.955057


**Selecting rows**

`loc` selects rows by label:

In [198]:
df.loc['A']

W    2.706850
X    0.628133
Y    0.907969
Z    0.503826
Name: A, dtype: float64

Or select based off of position instead of label with `iloc`. Positions start at 0, so `2` is the third row, C:

In [199]:
df.iloc[2]

W   -2.018168
X    0.740122
Y    0.528813
Z   -0.589001
Name: C, dtype: float64

**Selecting a subset of rows and columns**

Give `loc` the row label(s) first, then the column label(s). A single value comes back as a NumPy number such as `np.float64(...)`.

In [200]:
df.loc['B','Y']

np.float64(-0.8480769834036315)

In [201]:
df.loc[['A','B'],['W','Y']]

,W,Y
A,2.706850,0.907969
B,0.651118,-0.848077


### Conditional Selection

An important feature of pandas is conditional selection using bracket notation, very similar to NumPy. `df>0` gives a table of True/False values; putting it inside `df[...]` keeps the values where it is True and shows `NaN` elsewhere. A condition on one column, such as `df['W']>0`, keeps or drops whole rows.

In [202]:
df

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
C,-2.018168,0.740122,0.528813,-0.589001
D,0.188695,-0.758872,-0.933237,0.955057
E,0.190794,1.978757,2.605967,0.683509


In [203]:
df>0

,W,X,Y,Z
A,True,True,True,True
B,True,False,False,True
C,False,True,True,False
D,True,False,False,True
E,True,True,True,True


In [204]:
df[df>0]

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,NaN,NaN,0.605965
C,NaN,0.740122,0.528813,NaN
D,0.188695,NaN,NaN,0.955057
E,0.190794,1.978757,2.605967,0.683509


In [205]:
df[df['W']>0]

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
D,0.188695,-0.758872,-0.933237,0.955057
E,0.190794,1.978757,2.605967,0.683509


In [206]:
df[df['W']>0]['Y']

A    0.907969
B   -0.848077
D   -0.933237
E    2.605967
Name: Y, dtype: float64

In [207]:
df[df['W']>0][['Y','X']]

,Y,X
A,0.907969,0.628133
B,-0.848077,-0.319318
D,-0.933237,-0.758872
E,2.605967,1.978757


For two conditions use `&` (and) or `|` (or), with each condition in parentheses. Python's `and` and `or` do not work here, because they cannot combine whole columns of True/False values:

In [208]:
df[(df['W']>0) & (df['Y'] > 1)]

,W,X,Y,Z
E,0.190794,1.978757,2.605967,0.683509


**Try it:** which rows have `W` greater than 0 **or** `Y` greater than 1? Write the condition with `|`.

<details><summary>Answer</summary>

`df[(df['W']>0) | (df['Y'] > 1)]` returns rows A, B, D and E. Row C is the only one with a negative `W`, and its `Y` (0.53) is not above 1.

</details>

## More Index Details

Let's discuss some more features of indexing, including resetting the index or setting it to something else. We'll also talk about index hierarchy!

In [209]:
df

,W,X,Y,Z
A,2.706850,0.628133,0.907969,0.503826
B,0.651118,-0.319318,-0.848077,0.605965
C,-2.018168,0.740122,0.528813,-0.589001
D,0.188695,-0.758872,-0.933237,0.955057
E,0.190794,1.978757,2.605967,0.683509


In [210]:
# Reset to default 0,1...n index (the old labels move into a column called 'index')
df.reset_index()

,index,W,X,Y,Z
0,A,2.706850,0.628133,0.907969,0.503826
1,B,0.651118,-0.319318,-0.848077,0.605965
2,C,-2.018168,0.740122,0.528813,-0.589001
3,D,0.188695,-0.758872,-0.933237,0.955057
4,E,0.190794,1.978757,2.605967,0.683509


Add a column of US state codes, then use it as the index with `set_index`. Like `drop`, `set_index` returns a new DataFrame unless you pass `inplace=True`.

In [211]:
newind = 'CA NY WY OR CO'.split()

In [212]:
df['States'] = newind

In [213]:
df

,W,X,Y,Z,States
A,2.706850,0.628133,0.907969,0.503826,CA
B,0.651118,-0.319318,-0.848077,0.605965,NY
C,-2.018168,0.740122,0.528813,-0.589001,WY
D,0.188695,-0.758872,-0.933237,0.955057,OR
E,0.190794,1.978757,2.605967,0.683509,CO


In [214]:
df.set_index('States')

,W,X,Y,Z
States,,,,
CA,2.706850,0.628133,0.907969,0.503826
NY,0.651118,-0.319318,-0.848077,0.605965
WY,-2.018168,0.740122,0.528813,-0.589001
OR,0.188695,-0.758872,-0.933237,0.955057
CO,0.190794,1.978757,2.605967,0.683509


In [215]:
df

,W,X,Y,Z,States
A,2.706850,0.628133,0.907969,0.503826,CA
B,0.651118,-0.319318,-0.848077,0.605965,NY
C,-2.018168,0.740122,0.528813,-0.589001,WY
D,0.188695,-0.758872,-0.933237,0.955057,OR
E,0.190794,1.978757,2.605967,0.683509,CO


In [216]:
df.set_index('States',inplace=True)

In [218]:
df

,W,X,Y,Z
States,,,,
CA,2.706850,0.628133,0.907969,0.503826
NY,0.651118,-0.319318,-0.848077,0.605965
WY,-2.018168,0.740122,0.528813,-0.589001
OR,0.188695,-0.758872,-0.933237,0.955057
CO,0.190794,1.978757,2.605967,0.683509


## Multi-Index and Index Hierarchy

Let us go over how to work with Multi-Index, first we'll create a quick example of what a Multi-Indexed DataFrame would look like:

In [253]:
# Index Levels: pair each outer label with an inner number, ('G1', 1), ('G1', 2), ...
outside = ['G1','G1','G1','G2','G2','G2']
inside = [1,2,3,1,2,3]
hier_index = list(zip(outside,inside))
hier_index = pd.MultiIndex.from_tuples(hier_index)

In [254]:
hier_index

MultiIndex([('G1', 1),
            ('G1', 2),
            ('G1', 3),
            ('G2', 1),
            ('G2', 2),
            ('G2', 3)],
           )

In [257]:
# 6 rows (one per index pair) x 2 columns of random numbers
df = pd.DataFrame(np.random.randn(6,2),index=hier_index,columns=['A','B'])
df

A         B
G1 1  0.302665  1.693723
   2 -1.706086 -1.159119
   3 -0.134841  0.390528
G2 1  0.166905  0.184502
   2  0.807706  0.072960
   3  0.638787  0.329646

Now let's show how to index this! For index hierarchy we use `df.loc[]`; if this was on the columns axis, you would just use normal bracket notation `df[]`. Calling one level of the index returns the sub-DataFrame:

In [260]:
df.loc['G1']

,A,B
1,0.302665,1.693723
2,-1.706086,-1.159119
3,-0.134841,0.390528


In [263]:
df.loc['G1'].loc[1]

A    0.302665
B    1.693723
Name: 1, dtype: float64

The two index levels have no names yet (`None`). Naming them lets you refer to a level by name.

In [265]:
df.index.names

FrozenList([None, None])

In [266]:
df.index.names = ['Group','Num']

In [267]:
df

A         B
Group Num                    
G1    1    0.302665  1.693723
      2   -1.706086 -1.159119
      3   -0.134841  0.390528
G2    1    0.166905  0.184502
      2    0.807706  0.072960
      3    0.638787  0.329646

`xs` (cross-section) selects from a MultiIndex in one step: pass a tuple to go down both levels, or `level=` to pick an inner label in every outer group.

In [270]:
df.xs('G1')

,A,B
Num,,
1,0.302665,1.693723
2,-1.706086,-1.159119
3,-0.134841,0.390528


In [271]:
df.xs(('G1',1))

A    0.302665
B    1.693723
Name: (G1, 1), dtype: float64

In [273]:
df.xs(1,level='Num')

,A,B
Group,,
G1,0.302665,1.693723
G2,0.166905,0.184502


**Try it:** how do you get the value in column `B` for group `G2`, number 2?

<details><summary>Answer</summary>

`df.loc[('G2', 2), 'B']` (or `df.loc['G2'].loc[2]['B']`) gives `np.float64(0.07295967531703869)`, about 0.073.

</details>

# Great Job!